In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import datetime
import ipynbname

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import pubmed, gpt

In [ ]:
DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\HPV\elicit'
IN_PATH = os.path.join(DIR, 'Elicit - title.xlsx')
OUT_PATH = os.path.join(DIR, 'Elicit - title.xlsx')
LOG_PATH = os.path.join(DIR, 'log.txt')

In [ ]:
orig_df = pd.read_excel(IN_PATH)
orig_df.columns

In [ ]:
orig_df.rename(columns={'Publication Year': 'Year'}, inplace=True)
orig_df.loc[:, 'abstract'] = np.nan

In [ ]:
eligibility = gpt.check_eligibility(orig_df, criteria=gpt.JUMALAH_CRITERIA)
eligibility.columns

In [ ]:
eligibility.to_excel(OUT_PATH, index=False)

## Add abstracts

In [ ]:
eligibility.columns

In [ ]:
eligibility.rename(columns={'reasoning': 'title_reasoning', 'decision': 'title_decision'}, inplace=True)
del eligibility['abstract']
eligibility.head()

In [ ]:
abstract_source = pd.read_excel(os.path.join(DIR, '1556_with_eligibility_imp.xlsx'))
abstract_source.columns

In [ ]:
eligibility.title_extract.isna().sum(), abstract_source.title_extract.isna().sum()

In [ ]:
eligibility.title_extract.dtype, abstract_source.title_extract.dtype

In [ ]:
abstract_source = abstract_source[['title_extract', 'abstract']]
abstract_source['title_extract'] = abstract_source['title_extract'].astype(str)
eligibility['title_extract'] = eligibility['title_extract'].astype(str)
eligibility = pd.merge(eligibility, abstract_source, on='title_extract')
eligibility.head()

In [ ]:
eligibility = gpt.check_eligibility(eligibility, criteria=gpt.JUMALAH_CRITERIA)
eligibility.columns

In [ ]:
eligibility.rename(columns={'reasoning': 'abstract_reasoning', 'decision': 'abstract_decision'}, inplace=True)

In [ ]:
eligibility.to_excel(OUT_PATH, index=False)

In [ ]:
eligibility.in_1500.mean()

## Confusion matrix

In [ ]:
eligibility.columns

In [ ]:
def confusion_matrix(decision, gt):
    df = pd.DataFrame({'decision': decision, 'truth': gt})
    df['val'] = 1
    res = df.groupby(['decision', 'truth']).sum().reset_index(drop=False)
    return res.pivot(index='decision', columns='truth', values='val').fillna(0)

In [ ]:
log_file = open(LOG_PATH, 'at')

In [ ]:
log_file.write(f'\n==================== {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")} ====================\n\n')
log_file.write(f'{ipynbname.path()}\n')
log_file.write(f'Input: {IN_PATH}\n')
log_file.write(f'Output: {OUT_PATH}\n')

In [ ]:
log_file.write(f'\nSystem prompt:\n\n{gpt.SYSTEM_PROMPT}\n\n')
log_file.write(f'\nCriteria:\n\n{gpt.JUMALAH_CRITERIA}\n\n')

In [ ]:
confusion = confusion_matrix(decision=eligibility.title_decision, gt=eligibility.in_1500)
confusion

In [ ]:
log_file.write(f'\nTitle Confusion:\n{confusion}\n\n')

In [ ]:
confusion = confusion_matrix(decision=eligibility.abstract_decision, gt=eligibility.in_1500)
confusion

In [ ]:
log_file.write(f'\nAbstract Confusion:\n{confusion}\n\n')

In [ ]:
log_file.close()